Refer to [this wiki](https://dev.azure.com/tokyoelectron/EnterpriseDataWarehouse/_wiki/wikis/EnterpriseDataWarehouse.wiki/851/Data-Quality-Framework) for more details

In [ ]:
import yaml
import json
import os
from pathlib import Path
import re
import snowflake.connector
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

class NoAliasDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True

### Inputs ! 
- Repository Path
- Schema Name of New Arch
- Manifest File path 
- User Email to Query Snowflake if Needed

### Automation For Adding Default Test Cases in Bronze, Silver and Gold.

In [ ]:
enterprise_repository = r''
schema = '' 
# uses the latest manifest from the prod box
# if used elsewhere, make sure you do a dbt compile or similar to make sure the manifest is most up to date
manifest_path = 'P:\\Shares\\astro_datautility\\dags\\dbt\\dbt_tel\\target\\manifest.json'
email_user = "stanley.x.correia@us.tel.com"


In [ ]:
enterprise_repository = Path(enterprise_repository) 
base_path = enterprise_repository.joinpath('astro_datautility','dags','dbt','dbt_tel')


In [ ]:
with open(manifest_path, 'r') as f:
    manifest = json.load(f)

layers = ['bronze','silver','gold']

for item in layers:

    models_path = f'models\\{schema}\\{item}\\'
   
    inputModels = {node['name']: 1 for _ , node in manifest['nodes'].items() if node['resource_type'] == 'model' and node['original_file_path'].startswith(models_path)}

    inputYamlPath = base_path.joinpath(f'{models_path}_models.yml')

    inputModelTests = [
        'recently_run',
        'has_data',
    ]
    if item == 'bronze':
        inputColumnsTests = [
            { 
                'name': '_FILEDATE',
                'description': 'Timestamp of ingestion job start',
                'tests': [
                    {
                        'dbt_expectations.expect_row_values_to_have_recent_data': {
                            'datepart': 'hours',
                            'interval': 24
                        },
                    }
                ]
            },
        ]
    else:
        inputColumnsTests = []

    if os.path.exists(inputYamlPath):
        with open(inputYamlPath,'r') as y:
            yml_file = yaml.safe_load(y)
    else:
        yml_file = {'models': []}

    # first loops through the existing yaml, making sure to leave existing configs untouched
    for idx, model in enumerate(yml_file.get('models', [])):
        existing_model_tests = model.get('tests', [])
        existing_columns = model.get('columns', [])
        # loops through the model tests to be added if not already there
        for new_model_test in inputModelTests:
            if new_model_test not in existing_model_tests:
                existing_model_tests.append(new_model_test)
        # loops through columns/tests to be added 
        for new_col in inputColumnsTests:
            col_existed = False
            # loops though the existing model column's configs
            for col_idx, col in enumerate(existing_columns):
                if new_col['name'] == col['name']:
                    col_existed = True
                    existing_col_tests = col.get('tests', [])
                    # loops through the column tests to be added if not already there
                    for new_col_test in new_col['tests']:
                        if new_col_test not in existing_col_tests:
                            existing_col_tests.append(new_col_test)
                    # sorts the column tests by their key
                    col['tests'] = sorted(existing_col_tests, key=lambda d: list(d.keys())[0] if isinstance(d, dict) else d)
            if not col_existed:
                existing_columns.append(new_col)
        # sorts the model tests by their key
        model['tests'] = sorted(existing_model_tests, key=lambda d: list(d.keys())[0] if isinstance(d, dict) else d)
        
        if 'columns' in model:         
            # temporarily removes the columns to ensure they are placed at the bottom of the model's yaml
            del model['columns']
        # sorts the columns by their name ensuring underscores are placed at the top
        model['columns'] = sorted(existing_columns, key=lambda d: d.get('name', '').lower().replace('_', 'a'*9))
        # we won't need to process this model again so it can be removed from inputModels
        if model['name'] in inputModels:
            del inputModels[model['name']]
        
    # add models that weren't already part of the yaml
    for model_name in inputModels:
        input_yml = {
            'name': model_name,
            'tests': sorted(inputModelTests, key=lambda d: list(d.keys())[0] if isinstance(d, dict) else d),
        }
        if inputColumnsTests:
            input_yml['columns'] = inputColumnsTests
        yml_file['models'].append(input_yml)

    # sorts all the models by their name
    yml_file['models'] = sorted(yml_file['models'], key=lambda d: d.get('name', ''))

    with open(inputYamlPath,'w') as c:
        yaml.dump(yml_file ,c,default_flow_style=False,Dumper=NoAliasDumper,sort_keys = False)

### Automation For Bronze Test Cases.

---

This automation attempts the following:

#### Refinement Logic for Freshness Test Automation

- **Check if** `FILEDATE`, `CAPTUREDATETIME`, or `LOADDATETIME` fields are available in the model.

- **If these fields are present:**
  - Analyze the data load pattern.
  - If the data loads **consistently** (e.g., daily), **retain** the freshness check.
  - If the data loads **inconsistently** (e.g., sporadic, non-daily loads), **remove** the freshness check (or adjust it based on a specific pattern if observed).

- **If none of the fields are found:**  
  - **Remove** the freshness check for that model.  

---

> **Note:**  
> This does **not** mean we are removing the tests permanently.  
> Freshness checks will be applied iteratively where the model behavior justifies it.

---


In [ ]:
connection_details = {
    'user' : email_user,
    'account' : "telus-global.privatelink",
    'client_session_keep_alive' : False,
    'warehouse' : "DEVELOPER_WH",
    'database' : 'BRONZE',
    'role' : 'SYSADMIN_DEV',
    'schema' : 'PUBLIC',
    'authenticator' : "externalbrowser"
}

connection = snowflake.connector.connect(**connection_details)

bronze_yaml_path = base_path / 'models' / schema  / 'bronze' / '_models.yml'
model_path = base_path / 'models' / schema  / 'bronze' 
_scr_path = base_path / 'models' /  '_src'

list_of_models = [
    os.path.join(model_path, model) for model in os.listdir(model_path)  
]

ENV = 'DEV'

pattern = r'''(?i)\bFROM\b\s+\{\{\s*source\(\s*["']([^"']+)["']\s*,\s*["']([^"']+)["']\s*\)\s*\}\}'''

In [ ]:
def search_file_name(list_of_models, file):
    return next(
        item for item in list_of_models
        if re.search(
            pattern = f'{file}',
            string =item
        )
    )

def get_fqn_and_query(text):

    splits = re.split(
        pattern = r'__',
        string = text 
    )

    if ENV != 'PROD':

        query = f"""
            select 
                COLUMN_NAME 
            FROM {splits[-3]}_{ENV}.INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME ILIKE '%{splits[-1]}' AND TABLE_SCHEMA = '{splits[-2]}' ;
        """

        return f"{splits[-3]}_{ENV}.{splits[-2]}.{splits[-1]}" , query
    
    else:
        query = f"""
            select 
                COLUMN_NAME 
            FROM {splits[-3]}.INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME ILIKE '%{splits[-1]}' AND TABLE_SCHEMA = '{splits[-2]}' ;
        """

        return f"{splits[-3]}.{splits[-2]}.{splits[-1]}" , query    

def search_in_src_filename(match):
    for item in os.listdir(_scr_path):
        with open(os.path.join(_scr_path,item),'r') as f:
            yaml_dict = yaml.safe_load(f.read()).get('sources')
            for schema in yaml_dict:
                if schema.get('name').lower() == match.group(1).lower():
                    wanted_schema = schema.get('schema').upper()
                    for dx in schema.get('tables'):
                        if dx.get('name').lower() == match.group(2).lower():
                            wanted_table = dx.get('identifier').upper()
                            return wanted_schema,wanted_table  

def query_to_check(column_, schema , Name, result, database):
    query = f'''
        with LZ_data AS ( 
            select DISTINCT 
                TO_DATE({column_}) AS {column_},
                YEAR(TO_DATE({column_})) AS year
            FROM {database}.{schema}.{Name}
            WHERE {column_} >= DATEADD(MONTH, -1, CURRENT_DATE) 
        ),

        daliy_run AS (
            select 
                year,
                {column_},
                DAYNAME({column_}) AS days,
                DATEDIFF(DAY, {column_},
                    LAG({column_}) OVER (PARTITION BY year ORDER BY {column_})
                ) AS DAY_DIFF,
                CASE 
                    WHEN DAY_DIFF = -1 THEN 'X'
                    WHEN DAY_DIFF IS NULL THEN 'X'
                    ELSE '' 
                end AS daily 
            FROM LZ_data 
        ),

        final_result AS (
            select 
                year, 
                COUNT(CASE WHEN daily = '' THEN 1 END) AS blank_count,
                COUNT(CASE WHEN daily <> '' THEN 1 END) AS non_blank_count,
                COUNT(*) AS total_runs,
                CASE 
                    WHEN COUNT(*) = 1 THEN 'GO'
                    WHEN COUNT(*) < 20 THEN 'STOP'
                    WHEN COUNT(CASE WHEN daily = '' THEN 1 END) > 5 
                        AND COUNT(CASE WHEN daily = '' THEN 1 END) >= COUNT(CASE WHEN daily <> '' THEN 1 END)
                    THEN 'STOP' 
                    ELSE 'GO' 
                end AS result
            FROM daliy_run
            GROUP BY year
            ORDER BY year DESC
        )      
    '''
    if result:
        return query + 'select RESULT FROM final_result;' 
    else:
        return query + 'select * FROM daliy_run order by _CAPTUREDATETIME desc limit 20;'   

In [ ]:
with open(bronze_yaml_path,'r') as f:    
    dictonary_of_bronze = yaml.safe_load(f.read())
    dictonary_of_bronze_edit = dictonary_of_bronze.copy()
    cursor = connection.cursor()
    list_of_items = []
    for item in dictonary_of_bronze_edit.get('models'):
        try:
            path_of_model = search_file_name(list_of_models,item.get('name'))
            dictonary = dict()
            dictonary['name'] = item.get('name')
            dictonary['tests'] = ['has_data', 'recently_run']

            name, query = get_fqn_and_query(item.get('name'))
            cursor.execute(query)
            columns_in_view = [ column for item in cursor.fetchall() for column in item ]

            if '_FILEDATE' in columns_in_view:
                cursor.execute(f"Select max(_FILEDATE) from {name};")
                results_date = [ column for item in cursor.fetchall() for column in item ]

                with open(path_of_model,'r') as f:
                    data_in_model = f.read()

                match = re.search(pattern, data_in_model)

                if match:

                    schema,Name = search_in_src_filename(match)
                    check_query = query_to_check('_FILEDATE', schema , Name, True, 'LZ_PROD')
                    cursor.execute(check_query)
                    results = [
                        column
                        for item in cursor.fetchall() 
                        for column in item
                    ]

                    if results and 'STOP' not in results  :
                        proper_name = re.sub(pattern=r'_DEV|_QA|_SIT', string=name, repl= '')

                        dictonary['columns'] = [
                            {
                                'name': '_FILEDATE', 
                                'description': f'Check the _FILEDATE column in the {proper_name} table to ensure it falls within the last 24-hour interval.', 
                                'tests': [
                                    {'dbt_expectations.expect_row_values_to_have_recent_data': {
                                        'datepart': 'hours', 'interval': 24
                                        }
                                    }
                                ]
                            }
                        ]
                        markdown = f'''
Processed Model => {name}
File Date for {name} is {results_date[-1].isoformat()}
Added _FILEDATE Check for Model {name}
Model Source is => {schema} {Name}
'''
                        print(markdown)  

                    else:
                        print('\n')
                        check_query = query_to_check('_FILEDATE', schema , Name, False, 'LZ_PROD')

            elif '_CAPTUREDATETIME' in columns_in_view:
                with open(path_of_model,'r') as f:
                    data_in_model = f.read()

                match = re.search(pattern, data_in_model)

                if match:
                    
                    check_query = query_to_check('_CAPTUREDATETIME', match.group(1) , match.group(2), True, 'LZ_HVR')
                    cursor.execute(check_query)
                    results = [
                        column
                        for item in cursor.fetchall() 
                        for column in item
                    ]

                    if results and 'STOP' not in results  :
                        dictonary['columns'] = [
                            {
                                'name': '_CAPTUREDATETIME', 
                                'description': f'Check the _CAPTUREDATETIME column in the {name} table to ensure it falls within the last 24-hour interval.', 
                                'tests': [
                                    {'dbt_expectations.expect_row_values_to_have_recent_data': {
                                        'datepart': 'hours', 'interval': 24
                                        }
                                    }
                                ]
                            }
                        ]
                        markdown = f'''
Processed Model => {name}
Added _CAPTUREDATETIME to {name}
Model Source is => { match.group(1)}  {match.group(2)}
'''
                        print(markdown)   

                    else:
                        print(f'''Model has Gaps in their Data Flow {name}''')
                        check_query = query_to_check('_CAPTUREDATETIME', match.group(1) , match.group(2), False, 'LZ_HVR')
                        df = pd.read_sql(check_query, connection)
                        print(df.to_markdown(index=False))
                        print('\n')
                    
                else:
                    print('Cannot Regex the Below SAPO Model Please Check')
                    print(path_of_model)

            elif 'SNAPSHOT_DATE' in columns_in_view:
                    print(f"Added SNAPSHOT_DATE Test to {name}")
                    dictonary['columns'] = [
                        {
                            'name': 'SNAPSHOT_DATE', 
                            'description': f'Check the SNAPSHOT_DATE column in the {name} table to ensure it falls within the last 24-hour interval.', 
                            'tests': [
                                {'dbt_expectations.expect_row_values_to_have_recent_data': {
                                    'datepart': 'hours', 'interval': 24
                                    }
                                }
                            ]
                        }
                    ]                
            else:
                print(f'Test Removed for {name}')  
                              
            list_of_items.append(dictonary)

        except Exception as e:
            print(e)

        

In [ ]:
dictonary_of_bronze['models'] = list_of_items

with open(bronze_yaml_path,'w') as f:
    yaml.safe_dump(dictonary_of_bronze, f, sort_keys=False)